# Practical Statistics for Data Scientists (Python)
# Chapter 3 — Statistical Experiments and Significance Testing

> **Author:** Satrio Arjuna Putra  
> **Reference:** *Practical Statistics for Data Scientists* — Bruce, Bruce & Gedeck (O'Reilly)

---

## 📋 Chapter Summary

Chapter 3 covers **hypothesis testing** — the formal statistical framework for drawing conclusions from data.

| Topic | Key Concepts |
|-------|-------------|
| **A/B Testing** | Comparing two treatments; randomised experiment |
| **Hypothesis Tests** | Null/alternative hypothesis, test statistic |
| **Resampling** | Permutation tests; no distributional assumptions |
| **p-values** | Probability of observing results at least as extreme, given H₀ |
| **t-tests** | Comparing means between groups |
| **Multiple Testing** | False discovery rate, Bonferroni correction |
| **ANOVA** | Comparing means across 3+ groups (F-statistic) |
| **Chi-Square Test** | Independence of categorical variables |

---


## 🔧 Setup & Imports

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

DATA_BASE = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"
WEB_PAGE  = DATA_BASE + "web_page_data.csv"
FOUR_SESS = DATA_BASE + "four_sessions.csv"
CLICK_RATES = DATA_BASE + "click_rates.csv"
IMANISHI  = DATA_BASE + "imanishi_data.csv"

np.random.seed(42)
print("Libraries loaded!")


---

## 1. A/B Testing

### 📚 Theory

An **A/B test** (randomised controlled experiment) compares two variants (A = control, B = treatment) to determine which performs better.

**Steps:**
1. Define a metric (e.g., click-through rate)
2. Randomly assign subjects to A or B
3. Collect data
4. Test whether observed difference is statistically significant

> **Key Insight:** Randomisation ensures groups are comparable. The A/B test is the gold standard for causal inference in product development.


In [ ]:
# Web page session time A/B test
web_page_data = pd.read_csv(WEB_PAGE)
print(web_page_data.head())
print("\nGroup statistics:")
print(web_page_data.groupby('Page')['Time'].describe())


In [ ]:
# Visualise A/B test results
ax = web_page_data.boxplot(by='Page', column='Time', figsize=(8, 4))
ax.set_xlabel('Page Variant')
ax.set_ylabel('Session Time')
ax.set_title('Session Time by Page Variant (A/B Test)')
plt.suptitle('')
plt.tight_layout()
plt.show()


---

## 2. Hypothesis Testing Framework

### 📚 Theory

**Null Hypothesis (H₀):** The default assumption — no effect, no difference.  
**Alternative Hypothesis (H₁):** The effect we are trying to detect.

**Test statistic:** A number summarising the evidence against H₀.

**Decision rule:** Reject H₀ if test statistic is extreme enough (beyond critical value).

**Type I Error (α):** Rejecting H₀ when it is true (false positive)  
**Type II Error (β):** Not rejecting H₀ when H₁ is true (false negative)  
**Power (1-β):** Probability of correctly detecting a real effect

| | H₀ True | H₀ False |
|--|---------|---------|
| **Reject H₀** | Type I Error (α) | Correct (Power) |
| **Fail to Reject** | Correct | Type II Error (β) |

> **Key Insight:** There's always a trade-off between Type I and Type II errors. Setting α = 0.05 means we accept a 5% chance of false positives.


---

## 3. Resampling / Permutation Tests

### 📚 Theory

A **permutation test** answers: "Could the observed difference have occurred by chance?"

**Algorithm:**
1. Combine all observations from both groups
2. Randomly permute (shuffle) group labels many times
3. Compute the statistic for each permutation
4. p-value = proportion of permuted statistics as extreme as observed

**Advantage:** No distributional assumptions (non-parametric).

> **Key Insight:** Permutation tests work for any statistic and don't require normality. They are especially useful with small samples.


In [ ]:
# Permutation test for A/B web page data
page_a = web_page_data[web_page_data['Page'] == 'Page A']['Time']
page_b = web_page_data[web_page_data['Page'] == 'Page B']['Time']

# Observed difference
obs_diff = page_b.mean() - page_a.mean()
print(f"Observed difference (B - A): {obs_diff:.4f}")

# Permutation test
combined = np.concatenate([page_a, page_b])
n_a = len(page_a)
perm_diffs = []

for _ in range(10000):
    shuffled = np.random.permutation(combined)
    perm_diffs.append(shuffled[:n_a].mean() - shuffled[n_a:].mean())

perm_diffs = pd.Series(perm_diffs)
p_value = (perm_diffs.abs() >= abs(obs_diff)).mean()
print(f"p-value (permutation test): {p_value:.4f}")

# Plot
perm_diffs.plot.hist(bins=40, figsize=(8, 4))
plt.axvline(obs_diff, color='red', linestyle='--', label=f'Observed = {obs_diff:.2f}')
plt.xlabel('Permuted Mean Difference')
plt.title('Permutation Distribution')
plt.legend()
plt.tight_layout()
plt.show()


---

## 4. p-values and Statistical Significance

### 📚 Theory

The **p-value** is the probability of observing a test statistic at least as extreme as the one computed, *assuming H₀ is true*.

$$p = P(|T| \geq |t_{obs}| \mid H_0)$$

**Interpretation:**
- Small p-value → evidence against H₀ → statistically significant
- Conventional threshold: **α = 0.05**

**Common misconceptions:**
- ❌ p-value ≠ probability that H₀ is true
- ❌ p < 0.05 ≠ practically important
- ❌ p > 0.05 ≠ H₀ is true

> **Key Insight:** Statistical significance ≠ practical significance. Always report effect sizes alongside p-values.


---

## 5. t-Tests

### 📚 Theory

The **t-test** compares means between groups when:
- Data is approximately normal (or n is large)
- Population variance is unknown

**One-sample t-test:** $t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}$  
**Two-sample t-test:** $t = \frac{\bar{x}_1 - \bar{x}_2}{s_p \sqrt{1/n_1 + 1/n_2}}$

> **Key Insight:** Use Welch's t-test (unequal variances) by default, as it is more robust than the equal-variance version.


In [ ]:
# Two-sample t-test on web page data
t_stat, p_val = stats.ttest_ind(page_a, page_b)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Statistically significant (α=0.05): {p_val < 0.05}")


---

## 6. Multiple Testing Problem

### 📚 Theory

When you run many tests simultaneously, the probability of getting *at least one* false positive rises:

$$P(\text{at least 1 FP}) = 1 - (1-\alpha)^m$$

For m=20 tests at α=0.05: P(at least 1 FP) ≈ 64%!

**Corrections:**
- **Bonferroni:** Use α/m as the threshold (very conservative)
- **Benjamini-Hochberg (FDR):** Control the false discovery rate

> **Key Insight:** Always adjust for multiple comparisons. This is a major source of false findings in scientific literature.


In [ ]:
# Multiple testing demonstration
np.random.seed(42)
# Simulate 20 tests where H0 is TRUE for all
p_values = [stats.ttest_ind(np.random.randn(50), np.random.randn(50)).pvalue
            for _ in range(20)]

print("Raw p-values (20 tests, all H0 true):")
print([f"{p:.3f}" for p in p_values])
print(f"\nFalse positives (p < 0.05): {sum(p < 0.05 for p in p_values)}")

# Bonferroni correction
alpha_bonferroni = 0.05 / 20
print(f"\nBonferroni threshold: {alpha_bonferroni:.4f}")
print(f"False positives after Bonferroni: {sum(p < alpha_bonferroni for p in p_values)}")


---

## 7. ANOVA (Analysis of Variance)

### 📚 Theory

**ANOVA** extends the t-test to compare means across **3 or more groups**.

**F-statistic:**
$$F = \frac{\text{Variance between groups}}{\text{Variance within groups}}$$

Large F → means differ more than expected by chance → reject H₀.

**One-way ANOVA assumption:** Observations are independent, approximately normal, with equal variance across groups.

> **Key Insight:** ANOVA tells you *if* any group differs, not *which* groups differ. Use post-hoc tests (Tukey HSD) for pairwise comparisons.


In [ ]:
# ANOVA: compare session times across 4 page variants
four_sessions = pd.read_csv(FOUR_SESS)
print(four_sessions.head())
print("\nGroup means:")
print(four_sessions.groupby('Page')['Time'].mean())


In [ ]:
# One-way ANOVA
groups = [four_sessions[four_sessions['Page'] == p]['Time']
          for p in four_sessions['Page'].unique()]

f_stat, p_val = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Significant difference between groups (α=0.05): {p_val < 0.05}")


In [ ]:
# Visualise ANOVA groups
four_sessions.boxplot(by='Page', column='Time', figsize=(8, 4))
plt.xlabel('Page Variant')
plt.ylabel('Session Time')
plt.title('Session Time Across 4 Page Variants (ANOVA)')
plt.suptitle('')
plt.tight_layout()
plt.show()


---

## 8. Chi-Square Test

### 📚 Theory

The **chi-square test** tests whether two **categorical variables** are independent.

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

Where O = observed count, E = expected count (under independence).

> **Key Insight:** Chi-square requires sufficient expected counts (typically ≥ 5 per cell). Use Fisher's exact test for small samples.


In [ ]:
# Chi-square test: click rates by page variant
click_rate = pd.read_csv(CLICK_RATES)
print(click_rate)
print()

# Contingency table
clicks = pd.crosstab(click_rate['Source'], click_rate['Yes'] > 0)
print("Contingency table:")
print(clicks)


In [ ]:
# Chi-square test
chi2, p_val, dof, expected = stats.chi2_contingency(clicks)
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"Independent (α=0.05): {p_val > 0.05}")


---

## ✅ Chapter 3 Key Takeaways

1. **A/B tests** are the gold standard for comparing treatments; randomisation ensures comparability.
2. **Permutation tests** work for any statistic and require no distributional assumptions.
3. **p-values** measure evidence against H₀, NOT the probability H₀ is true.
4. **Multiple testing** inflates false positives — always correct for it (Bonferroni or FDR).
5. **ANOVA** compares 3+ group means using the F-statistic.
6. **Chi-square** tests independence between categorical variables.
7. **Statistical significance ≠ practical importance** — always report effect sizes.

---
*Reference: Chapter 3, Practical Statistics for Data Scientists (O'Reilly)*
